# Notebook 01 — Experiment 1: Tokenization Results

Analyses token count differentials across naming conventions (dot, camelCase, snake_case, kebab-case)  
and replicates the actuarial cost model from Pereira (2026a) using empirical ratios.

**Prerequisites:** Run `make corpus && make exp1` before executing this notebook.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import wilcoxon

from utils import EXP1_RESULTS

sns.set_theme(style='whitegrid')
df = pd.read_csv(EXP1_RESULTS)
print(f'Loaded {len(df)} rows — tokenizers: {df["tokenizer"].unique().tolist()}')
df.head()

## 1.1 Descriptive Statistics

In [ ]:
NOTATIONS = ['dot', 'camelCase', 'snake_case', 'kebab_case']
token_cols = [f'{n}_tokens' for n in NOTATIONS]

for tok in df['tokenizer'].unique():
    sub = df[df['tokenizer'] == tok][token_cols]
    print(f'\n=== {tok} ===')
    print(sub.describe().round(3))

## 1.2 Figure 1 — Token Count Distributions

In [ ]:
tokenizers = df['tokenizer'].unique()
PALETTE = {'dot': '#e74c3c', 'camelCase': '#2ecc71', 'snake_case': '#3498db', 'kebab_case': '#f39c12'}

fig, axes = plt.subplots(1, len(tokenizers), figsize=(4 * len(tokenizers), 5), sharey=False)
if len(tokenizers) == 1:
    axes = [axes]

for ax, tok in zip(axes, tokenizers):
    sub = df[df['tokenizer'] == tok]
    data = [sub[f'{n}_tokens'].values for n in NOTATIONS]
    bp = ax.boxplot(data, tick_labels=NOTATIONS, patch_artist=True)
    for patch, n in zip(bp['boxes'], NOTATIONS):
        patch.set_facecolor(PALETTE[n])
        patch.set_alpha(0.7)
    ax.set_title(tok)
    ax.tick_params(axis='x', rotation=30)
    ax.set_ylabel('Token count' if tok == tokenizers[0] else '')

fig.suptitle('Figure 1 — Token count distributions by notation × tokenizer', y=1.02)
plt.tight_layout()
plt.show()

## 1.3 Figure 2 — Dot/camelCase Ratio Distribution (H1)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for tok in df['tokenizer'].unique():
    sub = df[df['tokenizer'] == tok].copy()
    sub['ratio'] = sub['dot_tokens'] / sub['camelCase_tokens']
    ax.hist(sub['ratio'], bins=20, alpha=0.5, label=tok, density=True)

ax.axvline(1.67, color='black', linestyle='--', label='Theoretical 1.67× (Pereira 2026a)')
ax.axvline(1.0, color='grey', linestyle=':', label='Ratio = 1 (no overhead)')
ax.set_xlabel('dot / camelCase token ratio')
ax.set_ylabel('Density')
ax.set_title('Figure 2 — Inter-notation ratio distribution')
ax.legend()
plt.tight_layout()
plt.show()

## 1.4 Wilcoxon Signed-Rank Tests (H1)

In [ ]:
rows = []
for tok in df['tokenizer'].unique():
    sub = df[df['tokenizer'] == tok]
    stat, p = wilcoxon(sub['dot_tokens'], sub['camelCase_tokens'])
    ratio = (sub['dot_tokens'] / sub['camelCase_tokens']).mean()
    rows.append({'tokenizer': tok, 'mean_ratio': round(ratio, 4),
                 'W': round(stat, 2), 'p_value': round(p, 6),
                 'significant': p < 0.05})

pd.DataFrame(rows)

## 1.5 Cost Projection (H5)

In [ ]:
from cost_model import run as cost_run

for tok in df['tokenizer'].unique():
    result = cost_run(tokenizer=tok)
    ci = 95
    print(f"{tok}: ${result['point_estimate_usd']:,.2f} "
          f"(95% CI: ${result[f'ci_{ci}_lo_usd']:,.2f}–${result[f'ci_{ci}_hi_usd']:,.2f})")